# Week 1 — Foundations and Financial Data

FIN 43800 · Machine Learning in Finance

Andreas Neuhierl

# What is Machine Learning?

## Two Cultures

Leo Breiman, *Statistical Modeling: The Two Cultures* (2001):

- **Data modeling** — assume a stochastic model, estimate its
  parameters. Goal: describe *nature’s mechanism*.
- **Algorithmic modeling** — the mechanism is unknown; judge a model
  only by how well it predicts. Goal: *accuracy*.

## Which One Is This Course?

**Both, and you must know which you are doing.**

- “What is Apple’s beta?” — explanation. You want the parameter itself.
- “What will the market return next month?” — prediction. Nobody cares
  what the coefficients are if the forecast is good.

This course is mostly the second. The organizing question for fourteen
meetings: **are you explaining, or predicting?**

# What You Already Know

## Assumed: CAPM and Portfolio Basics

This course assumes you have met these. We are not re-teaching them.

- **Diversification** — combining imperfectly correlated assets lowers
  portfolio variance without lowering expected return proportionally.
- **Systematic vs. idiosyncratic risk** — only the part you cannot
  diversify away should earn a premium.
- **CAPM** — an asset’s expected excess return is proportional to its
  beta: $E[r_i - r_f] = \beta_i \times E[r_m - r_f]$.
- **Beta** — the slope from regressing an asset’s excess return on the
  market’s.

If any of that is unfamiliar, review this ASAP!

# What a Statistical Model Is

## The Market Model

$$ r_i = \alpha + \beta \, r_m + \varepsilon $$

Both returns are **excess** returns — already net of the risk-free rate.

|  |  |  |
|------------------------|------------------------|------------------------|
| **Model** | $r_i = \alpha + \beta r_m + \varepsilon$ | the relationship we assume |
| **Parameters** | $\alpha, \beta$ | unknown, fixed, never observed |
| **Estimates** | $\widehat\alpha, \widehat\beta$ | computed from a sample |
| **Fitted values** | $\widehat r_i = \widehat\alpha + \widehat\beta r_m$ | what the model says |
| **Residuals** | $r_i - \widehat r_i$ | what it missed |

## Three Words for the Same Thing

$r_m$ is a **predictor**. It is also a **feature**. It is also a
**regressor**.

- **Regressor** — econometrics. You will see it in the finance
  literature.
- **Predictor** — statistics. Used throughout ESL and ISLP.
- **Feature** — machine learning. Used in scikit-learn’s documentation.

Same object, three communities. This course uses all three,
deliberately, because you will meet all three.

## Where the Code Runs: Google Colab

**github.com/aneuhierl/ml-in-finance-fall2026** — every notebook, with
an “Open in Colab” link. The same link is on Brightspace. This week:
[open the week 1 notebook in
Colab](https://colab.research.google.com/github/aneuhierl/ml-in-finance-fall2026/blob/main/notebooks/w01-foundations.ipynb).

- **You need:** a **personal Google account** (Gmail) — not your Purdue
  login — and a laptop with a current Chrome, Firefox or Safari. Nothing
  to install; it is free.
- **Once per notebook:** File → **Save a copy in Drive**, then work in
  your copy. Edits to the original are not saved.
- **Every session:** download the data file from **Brightspace**, click
  the folder icon on Colab’s left edge, and drag the file in. Colab
  deletes it when the session ends.

## Optional: Python on Your Own Laptop

Colab is all this course requires. If you would rather work offline:

- **Install Anaconda** — free for coursework:
  [anaconda.com/download](https://www.anaconda.com/download) (“Skip
  registration” is fine). Needs about 5 GB of disk.
  - **Windows 11:** [install
    guide](https://www.anaconda.com/docs/getting-started/anaconda/install/windows-gui-install).
    Choose “Just Me”; leave “Add to PATH” unchecked.
  - **Mac:** [install
    guide](https://www.anaconda.com/docs/getting-started/anaconda/install/mac-gui-install).
    The current installer is for Apple-silicon Macs only.
- **Get the notebooks:** on the course GitHub page, Code → Download ZIP,
  and unzip it. Put Brightspace data files in a folder named `data` next
  to the notebook.
- **Open a notebook** from JupyterLab, launched from Anaconda Navigator.
- **One extra package:** run `pip install yfinance` once. (Later, for
  the text week: `conda install gensim`, not pip.)

## Two Return Conventions

Keep both straight, all semester:

- **Simple** returns, $P_t/P_{t-1} - 1$ — not **log** returns,
  $\ln(P_t/P_{t-1})$.
- **Excess** returns, net of the risk-free rate — not **gross** returns,
  with the risk-free rate still included.

The panel we load next is **simulated** — one market factor plus 25
assets — so we know the truth it came from and can check whether our
methods recover it.

## Setup

Run this cell first — after dragging `returns.csv` from Brightspace into
Colab. On Monday you will also upload `gw_monthly.csv` and
`gw_annual.csv`.

In [ ]:
import sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Fetch the one helper file we need from the course repo.
    import subprocess
    subprocess.run(
        ["wget", "-q", "-O", "mlfin_data.py",
         "https://raw.githubusercontent.com/aneuhierl/ml-in-finance-fall2026/main/src/mlfin_data.py"],
        check=True,
    )
else:
    # Find src/mlfin_data.py by walking UP from this notebook's folder, so the
    # import works wherever the notebook was saved (fresh clone: ../src).
    here = Path.cwd().resolve()
    src_folders = [folder / "src" for folder in [here, *here.parents]]
    matches = [folder for folder in src_folders if (folder / "mlfin_data.py").is_file()]
    sys.path.insert(0, str(matches[0]) if matches else "../src")

from mlfin_data import load_returns, resolve_data_path
# Simple monthly EXCESS returns in decimal form: 0.01 means one percent.
returns = load_returns()
print(returns.shape)
returns.iloc[:3, :4]

## Our first model: one asset on the market

In [ ]:
from sklearn.linear_model import LinearRegression

# X must be 2-dimensional for sklearn, even with a single predictor.
X = returns[["market"]].values
y = returns["asset_01"].values

model = LinearRegression().fit(X, y)

print(f"beta:  {model.coef_[0]:.3f}")
print(f"alpha: {model.intercept_:.5f}")
print(f"R^2:   {model.score(X, y):.3f}")

## The picture

In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(9, 5))
axis.scatter(returns["market"], returns["asset_01"], s=12, alpha=0.5)

line_x = [returns["market"].min(), returns["market"].max()]
line_y = [model.predict([[x]])[0] for x in line_x]
axis.plot(line_x, line_y, linewidth=2)

axis.set_xlabel("Market excess return (monthly)")
axis.set_ylabel("Asset 01 excess return (monthly)")
axis.set_title(f"Beta = {model.coef_[0]:.2f}")
plt.show()

# “Estimation Is Optimization”

## Choosing $\widehat\beta$

We have a model. How do we pick the numbers?

**Least squares:** pick the $\widehat\beta$ that makes the residuals
smallest.

$$ \widehat\beta = \arg\min_{b} \sum_{t} (r_{i,t} - a - b \, r_{m,t})^2 $$

- $\arg\min$ is doing all the work-
- Estimating a model means **solving an optimization problem**
- Python does it for us.

## Seeing the Minimum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

market = returns["market"].values
asset = returns["asset_01"].values
intercept = model.intercept_  # pinned at sklearn's own fit, so we can plot in 1-D

# Try many candidate slopes, in steps of 0.001, and record how badly each fits.
beta_grid = np.linspace(-0.5, 2.5, 3001)
sum_of_squares = [
    float(np.sum((asset - intercept - candidate * market) ** 2))
    for candidate in beta_grid
]

best = beta_grid[int(np.argmin(sum_of_squares))]

figure, axis = plt.subplots(figsize=(9, 5))
axis.plot(beta_grid, sum_of_squares, linewidth=2)
axis.axvline(best, linestyle="--")
axis.set_xlabel("candidate slope $b$", fontsize=13)
axis.set_ylabel("sum of squared residuals", fontsize=13)
axis.set_title(f"minimized at b = {best:.3f}", fontsize=15)
axis.tick_params(labelsize=11)
plt.show()

## Getting Up To Speed

**No prior Python is assumed.**

- **ISLP — *An Introduction to Statistical Learning with Python*.** Free
  PDF: <https://www.statlearning.com/resources-python>. This is the book
  to read.
- **The ISLP labs**, runnable chapter by chapter:
  <https://github.com/intro-stat-learning/ISLP_labs/tree/stable>

<!-- -->

- **DataCamp**, free through Purdue. Optional, but the fastest way in if
  you have never written a line of Python — the invite link is on
  Brightspace; join with your @purdue.edu address.

<!-- -->

- **And AI.** See the policy: you are expected to use it.

# In Sample and Out of Sample

## The Only Question That Matters

- Fit on data you have. Judge on data you did **not**.
- This is the criterion for **prediction**.
- For explanation — “what *is* Apple’s beta?” — the in-sample estimate
  itself is the object of interest.

In [ ]:
from sklearn.linear_model import LinearRegression

half = len(returns) // 2
X = returns[["market"]].values
y = returns["asset_01"].values

early = LinearRegression().fit(X[:half], y[:half])

print(f"in-sample  R^2, first {half} months: {early.score(X[:half], y[:half]):.3f}")
print(f"out-of-sample R^2, last {len(returns) - half} months: {early.score(X[half:], y[half:]):.3f}")

## In-Sample $R^2$ Is Not the Criterion

In [ ]:
import numpy as np

generator = np.random.default_rng(0)

# Add predictors that are PURE NOISE -- they cannot possibly help.
for how_many in (0, 5, 20, 50):
    noise = [generator.normal(size=len(X)) for _ in range(how_many)]
    X_padded = np.column_stack([X] + noise) if noise else X
    fit = LinearRegression().fit(X_padded, y)
    print(f"{how_many:2d} random predictors -> in-sample R^2 {fit.score(X_padded, y):.3f}")

- In-sample $R^2$ **never goes down** when you add a predictor. Not even
  a useless one.
- Modern models often have **very many** features — hundreds of firm
  characteristics, thousands of words. In-sample fit is then
  meaningless.

# The Curse of Dimensionality

## The Same Data, Spread Thinner

Twenty points, spread uniformly over $[0, 2]$ in each dimension. How
many land in the unit bin $[0, 1]$?

In [ ]:
import numpy as np

# Seed 23 reproduces the counts in Parsons, Haque & Liu (2004), Figure 1.
generator = np.random.default_rng(23)
points = generator.uniform(0, 2, size=(20, 3))


def count_in_unit_bin(points, n_dimensions):
    """How many points have every one of their first n coordinates in [0, 1]?"""
    count = 0
    for point in points:
        if max(point[:n_dimensions]) <= 1.0:
            count += 1
    return count


for n_dimensions in (1, 2, 3):
    count = count_in_unit_bin(points, n_dimensions)
    print(f"{n_dimensions} dimension(s): {count} of 20 points in the unit bin")

## Same Points, Pictured

Same 20 points. Blue ones sit inside the unit bin; gray ones do not.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 -- registers the 3-D projection

figure = plt.figure(figsize=(11, 5.5))

# Panel 1: one dimension -- the bin is the segment [0, 1] on a line.
axis_1 = figure.add_subplot(1, 3, 1)
for x, y, z in points:
    color = "tab:blue" if x <= 1.0 else "tab:gray"
    axis_1.scatter(x, 0, s=140, color=color)
axis_1.axvspan(0, 1, color="tab:orange", alpha=0.25)
axis_1.set_xlim(0, 2)
axis_1.set_yticks([])
axis_1.set_xlabel("dimension 1", fontsize=16)
axis_1.set_title(f"p = 1: {count_in_unit_bin(points, 1)} of 20 in bin", fontsize=20)
axis_1.tick_params(labelsize=14)

# Panel 2: two dimensions -- the bin is the unit square.
axis_2 = figure.add_subplot(1, 3, 2)
for x, y, z in points:
    inside = (x <= 1.0) and (y <= 1.0)
    color = "tab:blue" if inside else "tab:gray"
    axis_2.scatter(x, y, s=140, color=color)
axis_2.add_patch(plt.Rectangle((0, 0), 1, 1, color="tab:orange", alpha=0.25))
axis_2.set_xlim(0, 2)
axis_2.set_ylim(0, 2)
axis_2.set_xlabel("dimension 1", fontsize=16)
axis_2.set_ylabel("dimension 2", fontsize=16)
axis_2.set_title(f"p = 2: {count_in_unit_bin(points, 2)} of 20 in bin", fontsize=20)
axis_2.tick_params(labelsize=14)

# Panel 3: three dimensions -- the bin is the unit cube.
axis_3 = figure.add_subplot(1, 3, 3, projection="3d")
for x, y, z in points:
    inside = (x <= 1.0) and (y <= 1.0) and (z <= 1.0)
    color = "tab:blue" if inside else "tab:gray"
    axis_3.scatter(x, y, z, s=70, color=color)

# Draw the unit cube's 12 edges to mark the bin.
bottom_square = [(0, 0, 0), (1, 0, 0), (1, 1, 0), (0, 1, 0), (0, 0, 0)]
top_square = [(0, 0, 1), (1, 0, 1), (1, 1, 1), (0, 1, 1), (0, 0, 1)]
for square in (bottom_square, top_square):
    square_x = [corner[0] for corner in square]
    square_y = [corner[1] for corner in square]
    square_z = [corner[2] for corner in square]
    axis_3.plot(square_x, square_y, square_z, color="tab:orange", alpha=0.6)
for x, y in ((0, 0), (1, 0), (1, 1), (0, 1)):
    axis_3.plot([x, x], [y, y], [0, 1], color="tab:orange", alpha=0.6)

axis_3.set_xlim(0, 2)
axis_3.set_ylim(0, 2)
axis_3.set_zlim(0, 2)
axis_3.set_xlabel("dim 1", fontsize=16, labelpad=12)
axis_3.set_ylabel("dim 2", fontsize=16, labelpad=12)
axis_3.set_zlabel("dim 3", fontsize=16, labelpad=8)
axis_3.set_title(f"p = 3: {count_in_unit_bin(points, 3)} of 20 in bin", fontsize=20)
axis_3.tick_params(labelsize=14)

plt.tight_layout()
plt.show()

## But Breiman Said Dimensionality Is a *Blessing*

He did. Section 10 of the same paper: *“Instead of reducing
dimensionality, increase it by adding many functions of the predictor
variables.”*

Both are true, and the difference is the whole course:

- The **curse** bites *local averaging*. Neighbors stop being near.
- The **blessing** is real for methods that do **not** average locally —
  trees and forests partition adaptively; support vector machines use
  margins.
- Those methods extract small pieces of information from many weak
  predictors.
- So the curse is not “avoid many predictors”. **It is the reason you
  need the methods we spend fourteen meetings on.**

# Where This Course Goes

## Almost Everything Here Is a Time Series

A **time series** is a variable observed repeatedly over time, in order,
where the order matters. `returns["market"]` is 360 monthly
observations, oldest first.

Order is important:

- You may only use the past to predict the future.
- Observations near each other in time are related in ways that random
  samples are not.
- Shuffling the rows destroys the thing you are trying to model.

## Two Axes, Fourteen Meetings

**How many series?**

- *Now:* one at a time. Predict the market’s return next month.
- *Later:* thousands at once — every stock in the cross-section,
  together.

**How many predictors?**

- *Now:* one. `r_m`.
- *Later:* hundreds of firm characteristics.

Machine learning earns its keep in the bottom-right corner: **many
series, many predictors**. That is also exactly where the curse would
kill local averaging — which is why the methods matter.

## Your turn

1.  Open the week 1 notebook in Colab, save a copy in Drive, drag in
    `returns.csv` from Brightspace, and run the first cell. Also
    download `gw_monthly.csv` and `gw_annual.csv` from Brightspace:
    Monday uses them.
2.  If anything breaks, paste the error into an AI assistant first —
    that is what it is for. The TA is the last resort, not the first.
3.  **Optional, and worth it:** estimate the beta of every asset in the
    panel and find the three highest. A loop is fine; clarity beats
    cleverness.

## Your turn

On a card, before you leave. No names needed.

1.  You add 50 pure-noise predictors to a regression. Does its in-sample
    $R^2$ go up or down, and why is that a problem?
2.  A price goes 100 → 90 → 100. What are the two simple returns, and
    why do they not cancel?

# Python, Time Series, and Forecasting Out of Sample

# Python, the Bare Minimum

## Upload Three Files, Then Run Setup

Starting a new Colab session? Colab deletes uploaded files when a
session ends. Download `returns.csv`, `gw_monthly.csv` and
`gw_annual.csv` from Brightspace, drag them into the Files pane (folder
icon on the left), then run the **Setup** cell at the top of this
notebook before the cells below.

## How a Notebook Runs

In [ ]:
monthly_return = 0.012     # click the cell, press Shift + Enter
monthly_return * 12        # the LAST line of a cell is displayed

- Cells run in the order **you** ran them. Stuck? *Runtime → Restart
  session and run all.*
- An error? Read the **last red line** first. `NameError` means the cell
  that creates that name has not run yet.

## Names, Numbers, Arithmetic

In [ ]:
price_last_month = 100.00
price_today = 103.50
dividend = 0.50
# SIMPLE return, dividend included, in decimals (0.04 = 4 percent)
simple_return = (price_today + dividend) / price_last_month - 1
print(f"return: {simple_return:.2%}")    # :.2% shows a percent
print(f"{1.01 ** 12 - 1:.3f}")           # ** is "to the power of"

- `=` stores a value (R’s `<-`). `==` asks whether two values are equal.
- `f"...{name}..."` puts a value inside text; `:.3f` shows 3 decimals.
- Never write a power as `^`: `2 ^ 3` is 1 in Python, with no error.

## Python Counts From Zero

**Predict first:** `monthly_returns[1]` prints what — 0.012, −0.034, or
an error?

In [ ]:
monthly_returns = [0.012, -0.034, 0.021, 0.008, 0.015, -0.006]  # a list
print(len(monthly_returns))      # how many entries
print(monthly_returns[1])        # position 1 is the SECOND entry
print(monthly_returns[0])        # position 0 is the first
print(monthly_returns[-1])       # -1 counts from the end: the last entry
print(monthly_returns[0:3])      # positions 0, 1, 2: the end, 3, is left out

## Loops, One Line at a Time

In [ ]:
wealth = 1.00                      # $1 invested at the start
wealth_path = []                   # an empty list; the loop fills it
for r in monthly_returns:          # r takes each month's return in turn
    wealth = wealth * (1 + r)      # simple returns compound by multiplying
    wealth_path.append(wealth)     # .append (a method) adds to the list
print(f"six-month return: {wealth - 1:.2%}")

- The colon and the 4-space indent **are** the syntax (R uses braces).
- Read, don’t write: `if` follows the same rule.

``` python
if r < 0:
    print("a losing month")
else:
    print("a winning month")
```

## Your Turn: Compound the Bills

Three months of T-bill returns. Fill in the one line, then run it.

``` python
bill_returns = [0.004, 0.004, 0.005]   # simple monthly returns, decimals
wealth = 1.00
for r in bill_returns:
    wealth = ________                  # what happens to wealth each month?
print(f"three-month return: {wealth - 1:.2%}")
```

## Compound the Bills: The Answer

In [ ]:
bill_returns = [0.004, 0.004, 0.005]   # simple monthly returns, decimals
wealth = 1.00
for r in bill_returns:
    wealth = wealth * (1 + r)
print(f"three-month return: {wealth - 1:.2%}")
print(f"added up instead:   {sum(bill_returns):.2%}")   # sum() adds a list

Adding the three returns gives 1.30%. Compounding gives a little more.

## Functions: A Recipe With a Name

In [ ]:
def annualize(monthly_mean, monthly_std):
    # mean scales with 12; standard deviation with the square root of 12
    annual_mean = monthly_mean * 12
    annual_std = monthly_std * 12 ** 0.5
    return annual_mean, annual_std         # two answers at once

mean_a, std_a = annualize(0.010, 0.050)     # generic numbers, two names
print(f"{mean_a:.3f}  {std_a:.3f}")

- `def` names the recipe; `return` hands back the answer.
- The Goyal–Welch premium we use later today is **never** annualized.

## Five Libraries, and the Dot

In [ ]:
import numpy as np                 # arrays of numbers    (R: a vector)
import pandas as pd                # tables with columns  (R: data.frame)
import matplotlib.pyplot as plt    # charts               (R: plot)
import statsmodels.api as sm       # regression tables    (R: summary(lm))
from sklearn.linear_model import LinearRegression   # fit, then predict

- `import pandas as pd` loads a library under a short name.
- A **method** is a function attached to an object, called with a dot:
  `premium.mean()` is R’s `mean(premium)`.

## Which Months?

`monthly_returns[1:3]` — how many entries, and which ones?

# The Linear Model, With Its Table

## The Same Model, With Inference

The panel was simulated with **zero alpha**. **Predict:** will the
`const` row show a *t* near 0, or above 2?

In [ ]:
X = sm.add_constant(returns["market"])      # a column of ones: the intercept
fit = sm.OLS(returns["asset_01"], X).fit()  # y first, then X
print(fit.summary().tables[1])              # just the coefficient block
print(f"R-squared: {fit.rsquared:.3f}")

- statsmodels needs `add_constant`; sklearn adds the intercept itself.
- Same line as sklearn’s. sklearn is built to predict; statsmodels
  reports the inference.

## Reading the Table, Row by Row

- **coef**: the estimate. A market excess return 1 point higher goes
  with an asset_01 return 0.7066 points higher.
- **std err**: how far that estimate would move in another sample
  (0.063).
- **t** = coef / std err. Here 11.296: far beyond 1.96, so we reject
  “the true slope is zero” at the 5% level.
- **P\>\|t\|**: how likely a *t* at least this far from zero would be if
  the true coefficient were zero.
- **\[0.025 0.975\]**: the 95% confidence interval, about coef ± 2 std
  err.
- **const**: *t* = −0.035, *p* = 0.972. No sign of alpha, as the
  simulation promised.
- **R-squared** 0.263: the share of asset_01’s variation the market
  explains **in this sample**.

## Holding the Others Fixed

asset_02 on its own “explains” asset_01. **Does it survive next to the
market?**

In [ ]:
y = returns["asset_01"]
alone = sm.OLS(y, sm.add_constant(returns["asset_02"])).fit()
both = sm.OLS(y, sm.add_constant(returns[["market", "asset_02"]])).fit()
t_alone = alone.tvalues["asset_02"]
t_both = both.tvalues["asset_02"]
print(f"asset_02 alone:               t = {t_alone:.1f}")
print(f"asset_02 next to the market:  t = {t_both:.1f}")
print(both.summary().tables[1])

- Alone, asset_02 stands in for the market both assets share.
- Next to it, asset_02’s slope **holds the market fixed**. By
  construction of our simulation, that effect is zero.

# A Time Series in a DataFrame

## A DataFrame Is a Table With Names

In [ ]:
# resolve_data_path (from Setup) finds the file wherever you uploaded it
monthly = pd.read_csv(resolve_data_path("gw_monthly.csv"))
# yyyymm is a number like 192712 (Dec 1927): make it the row label, a date
dates_as_text = monthly["yyyymm"].astype(str)
monthly.index = pd.to_datetime(dates_as_text, format="%Y%m")
print(monthly.shape)                          # (rows, columns)
monthly[["ret", "Rfree", "equity"]].tail(3)   # [[ ]] picks a smaller TABLE

## One Column Is a Series

In [ ]:
premium = monthly["equity"].dropna()   # [ ] picks ONE column
print("file starts", monthly.index[0].year)
print("premium starts", premium.index[0].year)
premium.describe()

- `equity` = `ret` − `Rfree`: the S&P 500’s **simple** return, dividends
  included, minus the T-bill return. The **excess** return, or equity
  premium: monthly, in decimals, never annualized.
- The file starts in 1871, the premium in 1926: `.dropna()` drops the
  empty months.

## Three Ways to Pick Rows

**Predict first:** since 1926, how many months lost more than 15%, and
when?

In [ ]:
print(len(premium.loc["2000-01":"2009-12"]))   # by DATE: both ends included
print(premium.iloc[0:3])                       # by POSITION: rows 0, 1, 2
premium[premium < -0.15]                       # by CONDITION

- `.loc` picks by label (a date). `.iloc` picks by position, “end left
  out”.

## New Columns, and Last Month’s Value

In [ ]:
# long-term government yield minus T-bill yield (annual rates, decimals)
monthly["term_spread"] = monthly["lty"] - monthly["tbl"]     # row by row
# a dictionary {"name": column, ...} builds a table, one column per name
lagged = pd.DataFrame({"premium": monthly["equity"],
                       "premium_last_month": monthly["equity"].shift(1)})
lagged.loc["1929-09":"1929-12"]

- `.shift(1)` moves every value **down one row**, so each month also
  holds last month’s value. Excel: `=B2` typed in row 3. R: `lag()`.
- A forecast for month *t* may use only what was known at the end of
  month *t* − 1.

## Why sklearn Wants Two Dimensions

In [ ]:
x = np.array([0.012, -0.034, 0.021])
print(x * 100)                          # arithmetic works entry by entry
print(x.shape, x.reshape(-1, 1).shape)  # -1: "as many rows as needed"
print(monthly["tbl"].shape)             # [ ]:   1-D, fine as y
print(monthly[["tbl"]].shape)           # [[ ]]: 2-D, what sklearn needs as X

- A **matrix** is a table of numbers. `.shape` reports (rows, columns).
- A 1-D `X` makes sklearn stop with an error: it wants two dimensions.

## Your Turn: Last Month’s T-bill Rate

Build the column a forecast is allowed to use:

``` python
monthly["tbl_last_month"] = monthly["tbl"].________   # which method?
monthly.loc["1929-09":"1929-12", ["tbl", "tbl_last_month"]]
```

## Last Month’s T-bill Rate: The Answer

In [ ]:
monthly["tbl_last_month"] = monthly["tbl"].shift(1)
monthly.loc["1929-09":"1929-12", ["tbl", "tbl_last_month"]]

`.loc[rows, columns]` picks both at once.

## One Row Up or One Row Down?

What would `.shift(-1)` put in the January 1929 row — and why would a
forecast built on it be cheating?

# Persistent Is Not Profitable

## The Linear Model, With a Time Lag

$$ y_{t+1} = a + b \, x_t + e_{t+1} $$

- The same model as last week’s
  $r_i = \alpha + \beta \, r_m + \varepsilon$, and the textbook’s
  $\widehat y = \beta_0 + \beta_1 x$, with one change: $x$ is dated
  **before** $y$.
- $t$ counts periods (months, years). $x_t$ is known at the end of
  period $t$, so it can forecast $y_{t+1}$.
- When $x$ is $y$’s own past, $b$ measures **memory**. $b$ near 1: the
  series is **persistent**, each period much like the last. $b = 0$: the
  past says nothing about the future — tested with the same
  *t*-statistic as before.

## The Same Regression, Twice

**Predict:** which has the bigger slope on its own last year: the T-bill
return, or the stock market’s excess return?

In [ ]:
annual = pd.read_csv(resolve_data_path("gw_annual.csv")).set_index("yyyy")
# Rfree: T-bill return. equity: stocks minus Rfree. SIMPLE returns, per year.
for column in ["Rfree", "equity"]:
    data = pd.DataFrame({"this_year": annual[column],
                         "last_year": annual[column].shift(1)})
    data = data.loc[1927:2024]
    X = sm.add_constant(data["last_year"])
    fit = sm.OLS(data["this_year"], X).fit()
    slope = fit.params["last_year"]
    t_stat = fit.tvalues["last_year"]
    r2 = fit.rsquared
    print(f"{column}: slope {slope:.2f}, t {t_stat:.2f}, R2 {r2:.2f}")

## Persistence In a Picture

In [ ]:
plt.rcParams["font.size"] = 14               # bigger text for the projector
plt.figure(figsize=(10, 5))
plt.plot(annual.loc[1926:2024, "equity"], color="tab:gray")
plt.plot(annual.loc[1926:2024, "Rfree"], color="tab:blue", linewidth=3)
plt.axhline(0, color="black", linewidth=0.8)   # a line at zero
plt.ylim(-0.65, 0.8)
plt.text(1927, 0.66, "gray: stock excess return", color="tab:gray")
plt.text(1927, -0.6, "blue: T-bill return", color="tab:blue")
plt.title("Bills wander slowly; the premium jumps around its average")
plt.xlabel("year")
plt.ylabel("return over the year (decimal)")
plt.show()

- Every chart you hand in needs a title and axis labels with units.

## Does Log D/P Predict Next Year?

**D/P** is dividends over price: high when stocks are cheap.
**Predict:** will log D/P’s *t* on next year’s premium clear 1.96?

In [ ]:
# D/P = 12-month dividends / price. annual has no price: use each December.
december = monthly[monthly.index.month == 12]
log_dp = np.log(december["d12"]) - np.log(december["Index"])   # ln(D/P)
log_dp.index = december.index.year     # label the rows by year, like annual
annual["log_dp"] = log_dp              # pandas lines the rows up by year
annual["log_dp_last_year"] = annual["log_dp"].shift(1)
sample = annual.loc[1927:2024]
X = sm.add_constant(sample["log_dp_last_year"])
for y_name in ["log_dp", "equity"]:    # its own memory, then the premium
    fit = sm.OLS(sample[y_name], X).fit()
    slope = fit.params["log_dp_last_year"]
    t_stat = fit.tvalues["log_dp_last_year"]
    r2 = fit.rsquared
    print(f"{y_name}: slope {slope:.3f}, t {t_stat:.2f}, R2 {r2:.3f}")

## This Regression Knew the Future

- The regression used 1927–2024 all at once. An investor in 1960 did not
  know 1961–2024.
- **Split by time, never at random.** Fit on the past; judge on what
  came after.
- A random split mixes the eras: a model trained partly on the 2000s
  would “forecast” 1950.
- For car models or restaurant bills, a random split is fine: no time
  order.

## Persistent, or Profitable?

1.  The T-bill’s own-lag $R^2$ is 0.83. Can you make money forecasting
    next year’s bill return? Why, or why not?
2.  Log D/P forecast next year’s premium with *t* = 1.44 over 1927–2024.
    Give one reason not to trade on it yet.

# Out of Sample, in Real Time

## The Investor Who Lives in 1960

- To forecast 1960, fit on 1927–1959 only. A year later, add 1960,
  refit, forecast 1961. The training window **expands** by one year each
  time.
- Every forecast uses only what an investor knew then: 65 forecasts,
  1960 to 2024.

## The Benchmark, and the Score

**The benchmark** is the **historical mean**: the average premium over
the same training years. It is the forecast of someone who believes
returns cannot be predicted.

**The score**, out of sample:

$$ R^2_{OOS} = 1 - \frac{\text{MSE}_{\text{model}}}{\text{MSE}_{\text{mean}}},
\qquad \text{MSE} = \text{the average of } (\text{actual} - \text{forecast})^2 $$

- **Positive:** the model’s forecasts beat the historical mean.
  **Negative:** they did worse.
- Not last week’s `.score` on the test half. Its benchmark was the test
  period’s own average, which nobody knew in advance.

## $R^2_{OOS}$ by Hand

Four years of returns, in percent. The historical mean from the earlier
years is 5, held fixed here to keep the arithmetic short.

| year | actual | historical mean | model A | model B |
|------|--------|-----------------|---------|---------|
| 1    | 10     | 5               | 12      | 15      |
| 2    | −5     | 5               | 2       | −10     |
| 3    | 20     | 5               | 9       | 0       |
| 4    | 3      | 5               | 6       | 10      |

MSE of the mean: $(5^2 + 10^2 + 15^2 + 2^2)/4 = 354/4 = 88.5$.

**On paper:** the MSE of model A, then its $R^2_{OOS}$.

## $R^2_{OOS}$ by Hand: The Answer

In [ ]:
actual = np.array([10, -5, 20, 3])         # percent
historical_mean = np.array([5, 5, 5, 5])
model_a = np.array([12, 2, 9, 6])
model_b = np.array([15, -10, 0, 10])
# arrays subtract entry by entry; ** 2 squares each entry
mse_mean = np.mean((actual - historical_mean) ** 2)
mse_a = np.mean((actual - model_a) ** 2)
mse_b = np.mean((actual - model_b) ** 2)
print(f"MSE: mean {mse_mean}, A {mse_a}, B {mse_b}")
print(f"R2_OOS: A {1 - mse_a / mse_mean:.1%}, B {1 - mse_b / mse_mean:.1%}")

- Model B moves more and misses badly. A bold forecast that misses loses
  to the plain average.

## The Loop, Part 1: Set the Table

In [ ]:
# Row t: the premium of year t, and log D/P at the end of year t - 1.
data = pd.DataFrame({"premium": annual["equity"],
                     "predictor": annual["log_dp_last_year"]})
data = data.loc[1927:2024]
first_row = data.index.get_loc(1960)   # the row of 1960, the first forecast
print(first_row, "training years before the first forecast")

- Three empty lists, `[]`, will each grow by one entry a year with
  `.append`: the regression’s forecast, the historical mean’s forecast,
  and what actually happened.

## The Loop, Part 2: One Year at a Time

In [ ]:
model_forecasts = []        # the regression's forecasts
mean_forecasts = []         # the historical mean's forecasts
actual_premiums = []        # what actually happened
for t in range(first_row, len(data)):   # range(a, b): a, a+1, ..., b-1
    training = data.iloc[:t]            # years BEFORE row t: no peeking
    X_train = sm.add_constant(training["predictor"])
    fit = sm.OLS(training["premium"], X_train).fit()
    predictor_now = data["predictor"].iloc[t]    # known a year earlier
    forecast = fit.params["const"] + fit.params["predictor"] * predictor_now
    model_forecasts.append(forecast)
    mean_forecasts.append(training["premium"].mean())
    actual_premiums.append(data["premium"].iloc[t])
print(len(actual_premiums), "forecasts")
print("from", data.index[first_row], "to", data.index[-1])

## Your Turn: Score It

**Before you run it:** positive or negative?

``` python
actual = np.array(actual_premiums)                 # lists -> arrays
mse_model = np.mean((actual - np.array(model_forecasts)) ** 2)
mse_mean = np.mean((actual - np.array(mean_forecasts)) ** 2)
r2_oos = 1 - ________ / ________     # the score from "The Benchmark"
print(f"R2_OOS: {r2_oos:.2%}")
```

## Score It: The Answer

In [ ]:
actual = np.array(actual_premiums)                 # lists -> arrays
mse_model = np.mean((actual - np.array(model_forecasts)) ** 2)
mse_mean = np.mean((actual - np.array(mean_forecasts)) ** 2)
r2_oos = 1 - mse_model / mse_mean
print(f"R2_OOS: {r2_oos:.2%}")

- Negative: from 1960 on, the log D/P forecasts lost to the plain
  historical mean.
- In sample the same predictor had $R^2$ 0.021. Out of sample, it did
  worse than guessing the average. Goyal and Welch (2008) find this for
  most predictors.

## Before our next meeting

1.  Read Cochrane (2011), *Discount Rates*, Section I. Goyal &
    Welch (2008) is required for class.
2.  Next session: upload `gw_monthly.csv` and `gw_annual.csv` again, and
    run Setup.

## Readings

**Required** (on Brightspace): Breiman (2001), *Statistical Modeling:
The Two Cultures*, Statistical Science 16(3).

**Optional:**

- Donoho (2000), *High-Dimensional Data Analysis: The Curses and
  Blessings of Dimensionality*.
- Parsons, Haque & Liu (2004), *Subspace Clustering for High Dimensional
  Data*, SIGKDD Explorations 6(1) — the source of the bin figure.
- Peyré, Delon & Fadili, *Curses and Blessings of High Dimension*:
  <https://mathematical-coffees.github.io/mc08-high-dimension/>
- ISLP, chapter 2, and section 3.1 (simple linear regression, and how to
  read its table).